# Phase 4 — Data Science Check
Train a RandomForest model with MLflow tracking.
Uses 1% sample to fit in local memory.

In [1]:
import os
os.chdir(r"D:\Retail Demand Forecasting")

In [2]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales
from retail_demand_forecasting.nodes.feature_engineering import create_features
from retail_demand_forecasting.nodes.data_science import train_model

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("phase4_data_science")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


## 1. Prepare features

In [3]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
calendar_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
sell_prices_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))

melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)
melted_df = melted_df.sample(fraction=0.01, seed=42)

params = {"lag_days": [7, 28], "rolling_window_days": [7, 28]}
featured_df = create_features(melted_df, params)

# Convert to pandas for sklearn
feature_pd = featured_df.toPandas()
print(f"Feature matrix: {feature_pd.shape[0]:,} rows x {feature_pd.shape[1]} cols")

Feature matrix: 1,417 rows x 27 cols


## 2. Train model

In [4]:
train_params = {
    "model_params": {"max_depth": 5, "num_trees": 50},
    "target_col": "sales",
    "test_size": 0.2,
    "random_state": 42,
}

metrics = train_model(feature_pd, train_params)

2026/08/16 16:05:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/08/16 16:07:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


## 3. Metrics

In [5]:
print(f"MAE:  {metrics['mae']:.4f}")
print(f"RMSE: {metrics['rmse']:.4f}")
print(f"MAPE: {metrics['mape']:.2f}%")
print(f"Features: {metrics['features']}")

MAE:  1.1070
RMSE: 1.8131
MAPE: 68.26%
Features: ['lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI', 'has_event_1', 'has_event_2', 'sell_price']


In [6]:
spark.stop()
print("Phase 4 complete.")

Phase 4 complete.
